# H-JEPA-LM Demo

This notebook demonstrates how to use the H-JEPA-LM library for:
1. Creating and inspecting models
2. Training on text data
3. Visualizing embedding diversity
4. Using the world model for planning

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from jepalm import JEPELM, JEPAConfig
from hjepa_model import HJEPELM, HConfig

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Create Models

In [ ]:
# Base JEPA-LM
config = JEPAConfig(
    enc_hidden_dim=256, enc_num_layers=4, enc_num_heads=4,
    pred_hidden_dim=128, pred_num_layers=2,
    dec_hidden_dim=256, dec_num_layers=2, dec_num_heads=4,
)
model = JEPELM(config)
n = sum(p.numel() for p in model.parameters())
print(f'JEPA-LM: {n:,} parameters')

# H-JEPA-LM
hconfig = HConfig(
    enc_dim=256, enc_layers=4, enc_heads=4,
    pred_dim=128, pred_layers=2,
)
hmodel = HJEPELM(hconfig)
n = sum(p.numel() for p in hmodel.parameters())
print(f'H-JEPA-LM: {n:,} parameters')

## 2. Forward Pass

In [ ]:
# Create dummy token IDs
x = torch.randint(0, 1000, (4, 64))

# Base model
with torch.no_grad():
    out = model(x)
print('JEPA-LM output keys:', list(out.keys()))
print(f'  z_pred shape: {out["z_pred"].shape}')
print(f'  z_target shape: {out["z_target"].shape}')

# H-JEPA-LM
with torch.no_grad():
    hout = hmodel(x)
print('\nH-JEPA-LM output keys:', list(hout.keys()))
print(f'  z_pred[0] shape: {hout["z_pred"][0].shape}')
print(f'  z_pred[1] shape: {hout["z_pred"][1].shape}')

## 3. Embedding Diversity Visualization

In [ ]:
# Generate embeddings from different inputs
sentences = [
    torch.randint(0, 1000, (1, 32)) for _ in range(20)
]

embeddings = []
with torch.no_grad():
    for s in sentences:
        z = model.target_encoder(s).mean(dim=1)
        embeddings.append(z.numpy())

embeddings = np.concatenate(embeddings, axis=0)

# Compute pairwise cosine similarities
from torch.nn.functional import cosine_similarity
e = torch.tensor(embeddings)
sims = []
for i in range(len(e)):
    for j in range(i+1, len(e)):
        sim = cosine_similarity(e[i:i+1], e[j:j+1]).item()
        sims.append(sim)

print(f'Mean cosine similarity: {np.mean(sims):.3f}')
print(f'Std: {np.std(sims):.3f}')

plt.hist(sims, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel('Cosine Similarity')
plt.ylabel('Count')
plt.title('Embedding Diversity (lower = more diverse)')
plt.axvline(np.mean(sims), color='red', linestyle='--', label=f'Mean: {np.mean(sims):.3f}')
plt.legend()
plt.show()

## 4. World Model Planning

In [ ]:
with torch.no_grad():
    # Encode some context
    z_context = model.target_encoder(torch.randint(0, 1000, (1, 32)))

    # Plan with H-JEPA-LM
    actions = torch.randn(1, 5, hconfig.enc_dim)  # 5 candidate actions
    planned = hmodel.plan(z_context, actions, num_rollouts=10)

print(f'Planned trajectory shape: {planned.shape}')
print('World model planning complete!')

## 5. Training Loop (Simplified)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
losses = []

for step in range(100):
    x = torch.randint(0, 1000, (8, 64))
    out = model(x)
    loss = out['jepa_loss']
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if (step + 1) % 20 == 0:
        print(f'Step {step+1}: loss={loss.item():.4f}')

plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('JEPA Loss')
plt.title('Training Loss')
plt.show()